# Notebook for symbolic hessian calculations

## Libraries

In [2]:
import sympy as sp
import numpy as np
from typing import Iterable


In [3]:
def compute_jacobian(func: sp.Basic, vars: Iterable[sp.Symbol]) -> sp.Matrix:
    """
    Computes the Jacobian matrix of a set of functions with respect to a set of variables.

    Parameters
    ----------
    func :
        Sympy expression representing the function.
    vars :
        List of sympy symbols representing the variables.

    Returns
    -------
    sympy.Matrix: The Jacobian matrix.
    """
    
    return sp.Matrix([sp.diff(func, var) for var in vars ])

In [4]:

def create_positions(number_of_particles: int, dimensionality : int = 3) -> list[sp.Symbol]:
    """
    Creates a list of sympy symbols representing positions' coordinates of particles in 3D space.

    Parameters
    ----------
    number_of_particles :
        Number of particles.

    Returns
    -------
    list[sp.Symbol]: List of sympy symbols representing particle positions.
    """
    if dimensionality == 1: 
        coordinates = ['x']
    elif dimensionality == 2:
        coordinates = ['x', 'y']
    else:
        coordinates = ['x', 'y', 'z']

    return [sp.symbols(f'{a}_{i}') for i in range(number_of_particles) for a in coordinates]

def obtain_mean_minimum_distance(positions: np.ndarray) -> float:
    """
    Computes the mean minimum distance between particles based on their positions.

    Parameters
    ----------
    positions :
        Initial positions of the particles as an (nparticles, dimensionality) array.

    Returns
    -------
    float: Mean distance between particles.
    """
    number_of_particles = positions.shape[0]
    dimensionality = positions.shape[1]
    
    if dimensionality not in [1, 2, 3]:
        raise ValueError("Dimensionality must be 1, 2, or 3.")
    
    minimum_distances = []
    for i in range(number_of_particles):
        distances = []
        for j in range(number_of_particles):
            if i != j:
                distance = np.linalg.norm(positions[i] - positions[j])
                distances.append(distance)
        minimum_distances.append(min(distances))
    return np.mean(minimum_distances)

def create_bonds(initial_positions: np.ndarray) -> list:
    """
    Creates a list of bonds between particles based on their initial positions. The bonds
    are represented as tuples containing indices of the particles and equilibrium distance.
    Only 1st nearest neighbors are considered.

    Parameters
    ----------
    initial_positions :
        Initial positions of the particles.

    Returns
    -------
    bonds:
        Tuples representing bonds between particles containing indices of the particles,
        spring constant, and equilibrium distance.
    """
    bonds = []
    number_of_particles = initial_positions.shape[0]
    minimum_distance = obtain_mean_minimum_distance(initial_positions)
    for i in range(number_of_particles):
        for j in range(i + 1, number_of_particles):
            distance = np.linalg.norm(initial_positions[i] - initial_positions[j])
            if distance < minimum_distance * 1.1:  # Only consider bonds within 10% of the minimum distance
                bonds.append([i, j, minimum_distance])
    return bonds

class elastic_energy:
    """
    Class to represent the elastic energy of a system.
    """
    
    def __init__(self, initial_positions: np.ndarray):
        """
        Initializes the elastic energy class.
        """
        self.number_of_particles = initial_positions.shape[0]
        self.dimensionality = initial_positions.shape[1]
        self.bonds = create_bonds(initial_positions)


    def sympy_expression(self) -> sp.Basic:
        """
        Computes the sympy expression for the elastic energy of the system.

        Parameters
        ----------
        positions : 
            List of sympy symbols representing the positions of particles.

        Returns
        -------
        sp.Basic: Sympy expression for the elastic energy.
        """
        
        positions = create_positions(self.number_of_particles, self.dimensionality)
        
        position_vectors = [
            sp.Matrix([positions[i * self.dimensionality + j] for j in range(self.dimensionality)])
            for i in range(self.number_of_particles)
        ]

        energy = 0
        for bond in self.bonds:
            i, j, equilibrium_distance = bond
            distance_vector = position_vectors[i] - position_vectors[j]
            distance = sp.sqrt(distance_vector.dot(distance_vector))
            energy += 0.5 * (distance - equilibrium_distance)**2
        
        energy *= sp.symbols('k')  # k is the spring constant, can be defined as a parameter
        return energy


In [5]:


def create_square_grid_positions(number_of_particles: int) -> np.ndarray:
    """
    Creates a square grid of positions for the particles.

    Parameters
    ----------
    number_of_particles :
        Number of particles.

    Returns
    -------
    positions:

    """
    side_length = int(np.sqrt(number_of_particles))
    positions = []
    for i in range(side_length):
        for j in range(side_length):
            positions.append(i)
            positions.append(j)
            positions.append(0)  # z-coordinate is 0 for 2D grid
    positions = np.array(positions).reshape((number_of_particles, 3))
    return positions


nparticles_test = 9
test_positons = create_square_grid_positions(nparticles_test)
minimum_distance = obtain_mean_minimum_distance(test_positons)
bonds = create_bonds(test_positons)
    
print(f"Minimum distance for {nparticles_test} particles in a square grid: {minimum_distance}")
print(f"Bonds for {nparticles_test} particles in a square grid:")
for bond in bonds:
    print(f"Particles {bond[0]} and {bond[1]} with equilibrium distance {bond[2]}")


Minimum distance for 9 particles in a square grid: 1.0
Bonds for 9 particles in a square grid:
Particles 0 and 1 with equilibrium distance 1.0
Particles 0 and 3 with equilibrium distance 1.0
Particles 1 and 2 with equilibrium distance 1.0
Particles 1 and 4 with equilibrium distance 1.0
Particles 2 and 5 with equilibrium distance 1.0
Particles 3 and 4 with equilibrium distance 1.0
Particles 3 and 6 with equilibrium distance 1.0
Particles 4 and 5 with equilibrium distance 1.0
Particles 4 and 7 with equilibrium distance 1.0
Particles 5 and 8 with equilibrium distance 1.0
Particles 6 and 7 with equilibrium distance 1.0
Particles 7 and 8 with equilibrium distance 1.0


In [ ]:

def pair_initial_positions()-> np.ndarray:
    """
    Creates a pair of initial positions for testing.

    Returns
    -------
    np.ndarray: Initial positions of the particles.
    """
    return np.array([[-0.5, 0, 0], [0.5, 0, 0]])  # Two particles at (-0.5, 0, 0) and (0.5, 0, 0)


number_of_particles = 2
positions = create_positions(number_of_particles)
initial_positions = pair_initial_positions()


elastic_en = elastic_energy(initial_positions)
function = elastic_en.sympy_expression()
print("\nElastic Energy Function:")
sp.pretty_print(sp.simplify(function))


jacobian = compute_jacobian(function, positions)
print("\nJacobian 0 element:")
sp.pretty_print(sp.simplify(jacobian[0]))

def compute_hessian(func: sp.Basic, vars: Iterable[sp.Symbol]) -> sp.Matrix:
    """
    Computes the Hessian matrix of a function with respect to a set of variables.

    Parameters
    ----------
    func :
        Sympy expression representing the function.
    vars :
        List of sympy symbols representing the variables.

    Returns
    -------
    sympy.Matrix: The Hessian matrix.
    """
    
    return sp.Matrix([[sp.diff(sp.diff(func, var1), var2) for var2 in vars] for var1 in vars])
hessian = compute_hessian(function, positions)
print("\nHessian 00 element:")
sp.pretty_print(sp.simplify(hessian[0, 0]))

#hessian_at_initial = 





Elastic Energy Function:
                                                       2
      ⎛   ______________________________________      ⎞ 
      ⎜  ╱          2            2            2       ⎟ 
0.5⋅k⋅⎝╲╱  (x₀ - x₁)  + (y₀ - y₁)  + (z₀ - z₁)   - 1.0⎠ 

Jacobian 0 element:
                ⎛   ______________________________________      ⎞
                ⎜  ╱          2            2            2       ⎟
1.0⋅k⋅(x₀ - x₁)⋅⎝╲╱  (x₀ - x₁)  + (y₀ - y₁)  + (z₀ - z₁)   - 1.0⎠
─────────────────────────────────────────────────────────────────
               ______________________________________            
              ╱          2            2            2             
            ╲╱  (x₀ - x₁)  + (y₀ - y₁)  + (z₀ - z₁)              

Hessian 00 element:
                           2                                                   ↪
            1.0⋅k⋅(x₀ - x₁)                                           1.0⋅k    ↪
───────────────────────────────────────── + 1.0⋅k - ────────────────────────── ↪

KeyboardInterrupt: 